# village-seg-baseline — does the satellite-trained model see anything in a drone frame?

The model from `probe.ipynb` learned on Esri tiles at **0.385 m/px**. The 2025
flight over Gonchary shot 1920 x 1080 frames at ~40 m AGL: ~50 x 26 m of ground,
so **~0.026 m/px — 15x finer**. Scale is the first thing that decides whether the
model can work at all, so every frame is predicted at several ground resolutions,
starting with the one it was trained at.

The same ground is also predicted **from the satellite**: the model runs over the
Esri z18 mosaic of the flight area and the frame's footprint is cut from that
prediction. That gives each drone prediction a partner to compare with — rough,
because the pose-derived footprint is only good to a few metres (the flight's
GPS, the 2025 camera calibration and a 50 m frame).

Frames are chosen **where the satellite prediction puts a track inside the
footprint**, spaced in time so the set is not one stretch of road. The ground is
scrubby pasture with dirt tracks (`pocs/drone-frame-boundaries/notes.md`); the
only mapped features are outside the flight footprint.

Inputs, all read-only:

- frames: `../archive/uav_localization_2025/dataset_utils/image_pairs_v2/*_drone.jpg`
- poses: `../archive/uav_localization_2025/dataset_utils/gps_data.csv` (same epoch-ms key)
- Esri mosaic: `pocs/drone-frame-boundaries/satelites/gonchary-esri-z18.jpg` + `gonchary-grid.json`
- camera: FOV 63 x 35 deg, altitude offset -13 m (2025 calibration)

`PYTORCH_ENABLE_MPS_FALLBACK=1 papermill pocs/village-seg-baseline/drone.ipynb pocs/village-seg-baseline/runs/drone-landcoverai.ipynb -p init landcoverai`

In [ ]:
# papermill parameters
SLUG = "village-seg-baseline"
init = "landcoverai"            # which checkpoint from probe.ipynb: model-<init>.pt
n_frames = 8
min_alt = 35.0                  # m AGL; lower frames are take-off / landing
min_gap_s = 20.0                # between chosen frames
gsds = [0.385, 0.19, 0.096]     # m/px to present the drone frame at; first = training GSD
frames = []                     # explicit frame timestamps (epoch ms) override the auto choice

## Setup

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import bisect, csv, json, math, warnings
from pathlib import Path

import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import segmentation_models_pytorch as smp

warnings.filterwarnings("ignore")
Image.MAX_IMAGE_PIXELS = None
assert Path("datasets").is_dir(), "run from src/ — cwd is " + str(Path.cwd())

from tools.annotator.geo import Grid
from tools.annotator.refs._infer import tiled_logits

HERE = Path("pocs") / SLUG
FIGS = HERE / "figures"
ARCHIVE = Path("..") / "archive/uav_localization_2025/dataset_utils"
SAT_DIR = Path("pocs/drone-frame-boundaries")
FOV_W_DEG, FOV_H_DEG, ALT_OFFSET = 63.0, 35.0, -13.0

CLASSES = ["background", "building", "track", "woody", "field", "water"]
TRACK = CLASSES.index("track")
PAL = np.array([[40, 40, 40], [230, 30, 30], [255, 215, 0], [20, 160, 40],
                [200, 140, 255], [30, 110, 255]], np.uint8)
DEV = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

if init == "imagenet":
    MEAN, STD = np.array([0.485, 0.456, 0.406], np.float32), np.array([0.229, 0.224, 0.225], np.float32)
else:
    MEAN, STD = np.full(3, 0.5, np.float32), np.full(3, 0.25, np.float32)

model = smp.Unet("resnet34", encoder_weights=None, classes=len(CLASSES))
model.load_state_dict(torch.load(HERE / f"model-{init}.pt", map_location="cpu"))
model = model.eval().to(DEV)
print("model", init, "| device", DEV)


def norm(img):
    return (np.asarray(img, np.float32) / 255.0 - MEAN) / STD


def colour(lab):
    return PAL[lab]


@torch.no_grad()
def predict_small(img):
    """Whole-image prediction for anything up to a few thousand px: reflect-pad
    to a multiple of 32 (U-Net's stride), predict once, crop back."""
    x = torch.from_numpy(norm(img).transpose(2, 0, 1))[None]
    h, w = x.shape[2:]
    H, W = max(64, -(-h // 32) * 32), max(64, -(-w // 32) * 32)
    x = F.pad(x, (0, W - w, 0, H - h), mode="reflect" if (W - w < w and H - h < h) else "replicate")
    return model(x.float().to(DEV))[0, :, :h, :w].argmax(0).cpu().numpy().astype(np.uint8)

## Stage 0 — frames and poses

Each frame joins the nearest telemetry row by epoch-ms (worst gap 100 ms in the
earlier PoC). Footprint from altitude and FOV: `w = 2h·tan(63°/2)`.

In [ ]:
rows = list(csv.DictReader(open(ARCHIVE / "gps_data.csv")))
gt = [int(r["TimeUS"]) for r in rows]
poses = []
for p in sorted((ARCHIVE / "image_pairs_v2").glob("*_drone.jpg")):
    ts = int(p.name.split("_")[1])
    i = bisect.bisect_left(gt, ts)
    i = min(range(max(0, i - 1), min(len(gt), i + 2)), key=lambda j: abs(gt[j] - ts))
    r = rows[i]
    alt = float(r["Altitude"]) + ALT_OFFSET
    poses.append(dict(path=p, ts=ts, lat=float(r["Latitude"]), lon=float(r["Longitude"]),
                      alt=alt, yaw=float(r["Yaw"]), join_ms=abs(gt[i] - ts),
                      w_m=2 * alt * math.tan(math.radians(FOV_W_DEG / 2)),
                      h_m=2 * alt * math.tan(math.radians(FOV_H_DEG / 2))))
air = [p for p in poses if p["alt"] > min_alt]
print(f"{len(poses)} frames, {len(air)} above {min_alt:.0f} m AGL; "
      f"worst join gap {max(p['join_ms'] for p in poses)} ms")
alts = np.array([p["alt"] for p in air])
print(f"AGL median {np.median(alts):.1f} m -> footprint ~{2*np.median(alts)*math.tan(math.radians(31.5)):.0f} m wide, "
      f"{2*np.median(alts)*math.tan(math.radians(31.5))/1920:.3f} m/px native")

## Stage 1 — the flight area from the satellite

The model over the whole 2048 x 2048 Esri mosaic (790 m, 0.386 m/px) — its own
training scale, so this is the in-domain view of the ground the drone flew over.

In [ ]:
gmeta = json.loads((SAT_DIR / "gonchary-grid.json").read_text())
grid = Grid(z=gmeta["z"], x0=gmeta["x0"], y0=gmeta["y0"], nx=gmeta["nx"], ny=gmeta["ny"])
mosaic = Image.open(SAT_DIR / "satelites/gonchary-esri-z18.jpg").convert("RGB")
MPP = grid.mpp()

with torch.no_grad():
    sat_pred = tiled_logits(model, norm(mosaic), 512, 384, len(CLASSES), DEV).argmax(0).astype(np.uint8)
print(f"mosaic {mosaic.size} at {MPP:.3f} m/px")
print("class share over the mosaic:", {c: f"{(sat_pred == i).mean():.1%}" for i, c in enumerate(CLASSES)})

In [ ]:
def rotated_crop(arr_img, pose, resample):
    """Footprint of `pose` cut from a mosaic-aligned PIL image and turned to the
    UAV's heading -- the crop convention checked visually in
    pocs/drone-frame-boundaries/figures/crop-registration-check.png."""
    cx, cy = grid.to_px(pose["lat"], pose["lon"])
    w_px, h_px = pose["w_m"] / MPP, pose["h_m"] / MPP
    pad = int(math.hypot(w_px, h_px) * 0.6) + 8
    box = (int(cx - pad), int(cy - pad), int(cx + pad), int(cy + pad))
    if box[0] < 0 or box[1] < 0 or box[2] > arr_img.width or box[3] > arr_img.height:
        return None
    patch = arr_img.crop(box).rotate(pose["yaw"], resample=resample)
    mx, my = patch.width / 2, patch.height / 2
    return patch.crop((int(mx - w_px / 2), int(my - h_px / 2), int(mx + w_px / 2), int(my + h_px / 2)))


sat_pred_im = Image.fromarray(sat_pred)
for p in air:
    c = rotated_crop(sat_pred_im, p, Image.NEAREST)
    p["sat_track"] = float((np.asarray(c) == TRACK).mean()) if c is not None else None

if frames:
    chosen = [p for p in air if p["ts"] in set(frames)]
else:
    cand = sorted((p for p in air if p["sat_track"]), key=lambda p: -p["sat_track"])
    chosen = []
    for p in cand:
        if all(abs(p["ts"] - q["ts"]) >= min_gap_s * 1000 for q in chosen):
            chosen.append(p)
        if len(chosen) == n_frames:
            break
    chosen.sort(key=lambda p: p["ts"])

in_view = [p for p in air if p["sat_track"] is not None]
print(f"{sum(p['sat_track'] > 0 for p in in_view)}/{len(in_view)} footprints contain satellite-predicted track")
for p in chosen:
    print(f"  {p['ts']}  alt {p['alt']:5.1f} m  yaw {p['yaw']:6.1f}  "
          f"footprint {p['w_m']:.0f}x{p['h_m']:.0f} m  sat track {p['sat_track']:.1%}")

In [ ]:
def footprint_corners(p):
    cx, cy = grid.to_px(p["lat"], p["lon"])
    w, h = p["w_m"] / MPP / 2, p["h_m"] / MPP / 2
    t = math.radians(-p["yaw"])      # PIL rotate(yaw) is CCW; the footprint turns the other way on the map
    R = np.array([[math.cos(t), -math.sin(t)], [math.sin(t), math.cos(t)]])
    pts = np.array([[-w, -h], [w, -h], [w, h], [-w, h], [-w, -h]]) @ R.T
    return pts[:, 0] + cx, pts[:, 1] + cy


track_xy = np.array([grid.to_px(p["lat"], p["lon"]) for p in air])
fig, axes = plt.subplots(1, 2, figsize=(18, 9))
axes[0].imshow(mosaic); axes[0].set_title("Esri z18 mosaic, flight track (cyan), chosen footprints (red)")
axes[1].imshow(colour(sat_pred)); axes[1].set_title(f"model prediction from the satellite ({init})")
for a in axes:
    a.plot(track_xy[:, 0], track_xy[:, 1], "c-", lw=.8)
    for k, p in enumerate(chosen):
        xs, ys = footprint_corners(p)
        a.plot(xs, ys, "r-", lw=1.5); a.text(xs[0], ys[0], str(k), color="r", fontsize=11, weight="bold")
    a.axis("off")
plt.tight_layout(); plt.savefig(FIGS / f"drone-{init}-overview.png", dpi=90, bbox_inches="tight"); plt.show()

## Stage 2 — the drone frames, at several ground resolutions

Each frame is downsampled to each GSD in `gsds` — a real downsample of 1920 px,
never an upsample — and predicted whole. At the training GSD a frame is only
~130 x 70 px, so the model gets very little context; the finer GSDs trade
context for detail it never saw in training. Neither is obviously right, which is
why all are shown.

In [ ]:
out = []
for k, p in enumerate(chosen):
    frame = Image.open(p["path"]).convert("RGB")
    rec = {"k": k, "ts": p["ts"], "alt": p["alt"], "sat_track": p["sat_track"], "by_gsd": {}}
    preds = {}
    for g in gsds:
        size = (max(32, round(p["w_m"] / g)), max(32, round(p["h_m"] / g)))
        im = frame.resize(size, Image.LANCZOS)
        preds[g] = (im, predict_small(im))
        rec["by_gsd"][g] = {c: float((preds[g][1] == i).mean()) for i, c in enumerate(CLASSES)}
    sat_img = rotated_crop(mosaic, p, Image.BICUBIC)
    sat_lab = np.asarray(rotated_crop(sat_pred_im, p, Image.NEAREST))
    # drone at training GSD vs satellite, on the satellite crop's pixel grid
    d = np.asarray(Image.fromarray(preds[gsds[0]][1]).resize(sat_lab.shape[::-1], Image.NEAREST))
    ut = ((d == TRACK) | (sat_lab == TRACK)).sum()
    rec["agree_with_sat"] = float((d == sat_lab).mean())
    rec["track_iou_with_sat"] = float(((d == TRACK) & (sat_lab == TRACK)).sum() / ut) if ut else None
    out.append((p, frame, preds, sat_img, sat_lab, rec))
    print(f"[{k}] " + "  ".join(f"{g:.3f} m/px: track {rec['by_gsd'][g]['track']:5.1%} "
                                 f"bg {rec['by_gsd'][g]['background']:5.1%}" for g in gsds)
          + f"  | vs sat: agree {rec['agree_with_sat']:.2f}, track IoU "
          + (f"{rec['track_iou_with_sat']:.2f}" if rec['track_iou_with_sat'] is not None else "-"))

## Look at it

Per frame: the frame itself, the prediction at each GSD, then the satellite crop
of the same footprint and the model's prediction on it. **The question to answer
by eye:** where the frame shows a dirt track, is there yellow on it — at any
scale — and does it line up with the satellite-side yellow?

In [ ]:
ncol = 1 + len(gsds) + 2
fig, axes = plt.subplots(len(out), ncol, figsize=(3.6 * ncol, 2.2 * len(out) + 1))
axes = np.atleast_2d(axes)
for r, (p, frame, preds, sat_img, sat_lab, rec) in enumerate(out):
    axes[r, 0].imshow(frame); axes[r, 0].set_ylabel(f"[{r}] {p['alt']:.0f} m", fontsize=9)
    for j, g in enumerate(gsds):
        im, lab = preds[g]
        axes[r, 1 + j].imshow(im); axes[r, 1 + j].imshow(colour(lab), alpha=.5)
        if r == 0: axes[r, 1 + j].set_title(f"drone @ {g:.3f} m/px\n({im.size[0]}x{im.size[1]} px)", fontsize=9)
    axes[r, -2].imshow(sat_img); axes[r, -1].imshow(sat_img); axes[r, -1].imshow(colour(sat_lab), alpha=.5)
    if r == 0:
        axes[r, 0].set_title("drone frame", fontsize=9)
        axes[r, -2].set_title("Esri, same footprint", fontsize=9)
        axes[r, -1].set_title("prediction from satellite", fontsize=9)
for a in axes.ravel(): a.set_xticks([]); a.set_yticks([])
handles = [plt.Rectangle((0, 0), 1, 1, color=PAL[c] / 255) for c in range(6)]
fig.legend(handles, CLASSES, loc="lower center", ncol=6)
plt.tight_layout(rect=(0, .03, 1, 1)); plt.savefig(FIGS / f"drone-{init}-frames.png", dpi=90, bbox_inches="tight"); plt.show()

In [ ]:
summary = {"init": init, "gsds": gsds, "frames": [rec for *_, rec in out],
           "track_frac_by_gsd": {g: float(np.median([rec["by_gsd"][g]["track"] for *_, rec in out])) for g in gsds},
           "median_agree_with_sat": float(np.median([rec["agree_with_sat"] for *_, rec in out]))}
(HERE / f"results-drone-{init}.json").write_text(json.dumps(summary, indent=1, default=str))
print(json.dumps({k: v for k, v in summary.items() if k != "frames"}, indent=1))

## Verdict

Written into `notes.md` after looking at `figures/drone-<init>-frames.png`.